# Region properties tables

Measure skimage `regionprops` for objects in one or more **segmentations** (label mask images) and zero or more **intensity images** (channels). Saves one CSV table per field of view, named after the first segmentation's file stem.

- one row per object (label), columns for all selected properties
- shape properties are measured per segmentation, intensity properties per segmentation x channel combination
- all property columns are suffixed with the segmentation / channel name (e.g. `area_nucleus`, `intensity_mean_nucleus_uhrf1`), so the column schema is stable regardless of how many segmentations / channels are configured
- objects present in one segmentation but not in another get `NaN` in the missing columns (outer join on label)
- provenance: input file paths (relative to `in_path`) and optionally named indices extracted from file names are added as columns

Special cases:

- **no intensity images** (`intensity_images = None`): shape-only measurements
- single segmentation and single intensity image: equivalent to the old `simple_regionprops` recipe (superseded here)

## Configuration forms

`segmentations` and `intensity_images` accept (forms can also be mixed within a list):

- `['name', 'subdirectory/file_pattern']`
- `{'name': ..., 'subdirectory': ..., 'file_pattern': ...}`

**NOTE:** keep all parameter values JSON-compatible (lists & dicts, **no tuples**), so the notebook can be parameterized with papermill (see `run-parameterized-nb.ipynb`). Tuples work when running the notebook directly, but papermill turns them into strings.


In [ ]:
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from natsort import natsorted
from tifffile import imread
from skimage.measure import regionprops_table


In [ ]:
def normalize_spec(specs):
    """Normalize a segmentation / intensity image specification to a list of dicts.

    Accepted forms (also mixed in a list):
      - [name, 'subdirectory/file_pattern'] (tuples work as well when running directly)
      - {'name': ..., 'subdirectory': ..., 'file_pattern': ...}
      - None or []
    """
    if specs is None:
        return []
    if isinstance(specs, dict):
        return [specs]
    if isinstance(specs, str):
        raise ValueError(f'bare string spec not supported, use [name, "subdirectory/file_pattern"]: {specs!r}')
    specs = list(specs)
    if not specs:
        return []
    # shorthand: 2-element sequence of strings -> [name, 'subdirectory/file_pattern']
    if all(isinstance(s, str) for s in specs):
        if len(specs) != 2:
            raise ValueError(f'expected [name, "subdirectory/file_pattern"], got {specs!r}')
        name, where = specs
        subdir, _, pattern = where.partition('/')
        return [{'name': name, 'subdirectory': subdir, 'file_pattern': pattern or '[!.]*.tif'}]
    return [normalize_spec(s)[0] for s in specs]


def extract_filename_idxs(stem, idxs):
    """Extract named integer indices from a filename stem, e.g. '_pos3' -> pos=3.

    Indices not present in the name are omitted.
    """
    return {idx: int(m.group(1))
            for idx in idxs
            if (m := re.search(f'_{idx}([0-9]+)', stem)) is not None}


def mode(arr):
    vals, counts = np.unique_counts(arr)
    return vals[np.argmax(counts)].item()

# whole-image statistics for image_wide_properties
image_props_funs = {
    'mode': mode,
    'mean': np.mean,
}


In [ ]:
in_path = '/Users/david/Desktop/test_symbionts/sd_crest'

# segmentation (label mask) specifications, see forms above
segmentations = [
    ['ring', 'segmentation_ring/[!.]*.tif'],
    ['background', 'segmentation_background/[!.]*.tif'],
]

# intensity image specifications, same forms as segmentations
# set to None (or []) for shape-only measurements
intensity_images = ['lamp1', 'tif_maxprojection/[!.]*ch0*.tif']

out_subdirectory = 'region_properties_multichannel'

# shape properties, measured per segmentation (columns suffixed with segmentation name)
shape_properties_to_include = ['label', 'area']

# intensity properties, measured per segmentation x channel (columns suffixed _<segmentation>_<channel>)
intensity_properties_to_include = ['intensity_mean']

# whole-image statistics per channel (keys of image_props_funs), [] disables
image_wide_properties = []

# named indices to extract from file stems -> columns '<spec_name>_<idx>' (e.g. nucleus_pos)
# [] disables
filename_idxs_to_extract = []


In [ ]:
# normalize specs
segmentations = normalize_spec(segmentations)
intensity_images = normalize_spec(intensity_images)

# label is needed as join key
if 'label' not in shape_properties_to_include:
    shape_properties_to_include = ['label'] + list(shape_properties_to_include)
if 'label' not in intensity_properties_to_include:
    intensity_properties_to_include = ['label'] + list(intensity_properties_to_include)

# sorted file lists per spec
mask_file_lists = [natsorted((Path(in_path) / s['subdirectory']).glob(s['file_pattern'])) for s in segmentations]
image_file_lists = [natsorted((Path(in_path) / s['subdirectory']).glob(s['file_pattern'])) for s in intensity_images]

# all specs must yield the same number of files, so they can be paired per field of view
n_fovs = len(mask_file_lists[0]) if mask_file_lists else 0
for spec, files in zip(segmentations, mask_file_lists):
    assert len(files) == n_fovs, f"segmentation '{spec['name']}': {len(files)} files, expected {n_fovs}"
for spec, files in zip(intensity_images, image_file_lists):
    assert len(files) == n_fovs, f"channel '{spec['name']}': {len(files)} files, expected {n_fovs}"

# output file names (stems of the first segmentation's files) must be unique per field of view
out_names = [f'{files.stem}_measurements.csv' for files in mask_file_lists[0]]
assert len(set(out_names)) == len(out_names), f'output file names not unique: {out_names}'


In [ ]:
# show file combinations per field of view for verification
n_fovs, mask_file_lists, image_file_lists


In [ ]:
def add_provenance(df, specs, files):
    """Add input file paths (relative to in_path) and extracted filename indices as columns."""
    for spec, f in zip(specs, files):
        df[f"{spec['name']}_file"] = f.relative_to(Path(in_path))
        for idx, val in extract_filename_idxs(f.stem, filename_idxs_to_extract).items():
            df[f"{spec['name']}_{idx}"] = val
    return df


def process_one(fov_idx):
    mask_files_i = [files[fov_idx] for files in mask_file_lists]
    image_files_i = [files[fov_idx] for files in image_file_lists]

    # output file name: stem of the first segmentation's file
    # (file names are unique within a directory, so this is unique per field of view)
    out_name = f'{mask_files_i[0].stem}_measurements.csv'

    df = None
    for spec, mask_file in zip(segmentations, mask_files_i):
        seg_name = spec['name']

        # shape properties per segmentation
        mask = imread(mask_file)
        dfi = pd.DataFrame(regionprops_table(mask, properties=shape_properties_to_include))
        dfi = dfi.rename(columns={c: f'{c}_{seg_name}' for c in dfi.columns if c != 'label'})
        df = dfi if df is None else pd.merge(df, dfi, on='label', how='outer')

        # intensity properties per segmentation x channel
        for channel_spec, image_file in zip(intensity_images, image_files_i):
            ch_name = channel_spec['name']
            img = imread(image_file)

            dfi = pd.DataFrame(regionprops_table(mask, img, properties=intensity_properties_to_include))
            dfi = dfi.rename(columns={c: f'{c}_{seg_name}_{ch_name}' for c in dfi.columns if c != 'label'})
            df = pd.merge(df, dfi, on='label', how='outer')

            # whole-image statistics per channel
            for prop in image_wide_properties:
                df[f'{prop}_{ch_name}'] = image_props_funs[prop](img)

    # provenance: input files & filename indices
    df = add_provenance(df, segmentations, mask_files_i)
    df = add_provenance(df, intensity_images, image_files_i)

    df.to_csv(outdir / out_name, index=None)
    return df


outdir = Path(in_path) / out_subdirectory
outdir.mkdir(exist_ok=True)

with ThreadPoolExecutor() as tpe:
    results = list(tpe.map(process_one, range(n_fovs)))


In [ ]:
# show first result table
results[0]
